# LeetCode 2154 — Keep Multiplying Found Values by Two bằng Python

**Link đề:** https://leetcode.com/problems/keep-multiplying-found-values-by-two/
**Độ khó:** Easy · **Chủ đề:** Array, Hash Table, Sorting, Simulation

```python
class Solution:
    def findFinalValue(self, nums: list[int], original: int) -> int:
        ...
```

## Mục tiêu học
1. Mô phỏng (simulate) quá trình bằng vòng **`while`** với điều kiện dừng rõ ràng.
2. Hiểu vì sao `x in list` chậm (`O(n)`) còn `x in set` nhanh (trung bình `O(1)`).
3. Đếm số vòng lặp tối đa: giá trị **nhân đôi** mỗi lần nên chỉ có khoảng `log2(1000)` ≈ 10 vòng.
4. Thêm hai cách sáng tạo: **sắp xếp rồi quét một lượt**, và **bitmask các lũy thừa của 2**.
5. Trace từng bước và kiểm thử đối chiếu nhiều cách.

## 1. Đề bài

Cho mảng `nums` và số `original`. Lặp lại:
- Nếu `original` **có trong** `nums` → gán `original = 2 * original`.
- Nếu không có → **dừng**.

Trả về giá trị cuối cùng của `original`.

| `nums` | `original` | Kết quả | Diễn biến |
|---|:-:|:-:|---|
| `[5,3,6,1,12]` | 3 | `24` | 3 có → 6 có → 12 có → 24 không có → dừng |
| `[2,7,9]` | 4 | `4` | 4 không có → dừng ngay |

**Constraints:** `1 <= len(nums) <= 1000`, `1 <= nums[i], original <= 1000`.

Chú ý: thứ tự trong `nums` **không quan trọng** — chỉ cần biết "có hay không".

In [1]:
nums, original = [5, 3, 6, 1, 12], 3
chain = [original]
while chain[-1] in nums:
    chain.append(chain[-1] * 2)
print("chuỗi giá trị:", chain, "-> đáp án", chain[-1])

chuỗi giá trị: [3, 6, 12, 24] -> đáp án 24


## 2. Nền tảng Python cần biết

### 2.1. Vòng `while` và điều kiện dừng

`while điều_kiện:` lặp **chừng nào** điều kiện còn đúng. Phải chắc rằng mỗi vòng làm điều kiện "tiến về" sai, nếu không sẽ lặp vô hạn. Ở bài này `original` tăng gấp đôi mỗi vòng, còn giá trị trong `nums` ≤ 1000 → chắc chắn dừng.

In [2]:
x = 1
steps = 0
while x <= 1000:
    x *= 2
    steps += 1
print("x =", x, "sau", steps, "lần nhân đôi")

x = 1024 sau 10 lần nhân đôi


### 2.2. `x in list` — dò tuần tự

Python kiểm tra từng phần tử từ đầu đến khi gặp `x` hoặc hết list → tốn tới `n` phép so sánh.

In [3]:
def linear_search_steps(lst, x):
    for i, v in enumerate(lst):
        if v == x:
            return i + 1
    return len(lst)

nums = [5, 3, 6, 1, 12]
for x in (5, 12, 24):
    print(f"{x} in nums? {x in nums} | số phép so sánh: {linear_search_steps(nums, x)}")

5 in nums? True | số phép so sánh: 1
12 in nums? True | số phép so sánh: 5
24 in nums? False | số phép so sánh: 5


### 2.3. `set` — tra cứu nhanh nhờ băm

`set(nums)` tạo tập hợp; `x in s` tính **hash** của `x` để nhảy thẳng tới "ô" chứa nó → trung bình `O(1)`, không phụ thuộc kích thước set. Xây set tốn `O(n)` một lần.

In [4]:
s = set([5, 3, 6, 1, 12])
print(s)
print(6 in s, 24 in s)
print("hash(6) =", hash(6), "| hash('a') khác nhau giữa các lần chạy, nhưng hash số nguyên nhỏ = chính nó")

{1, 3, 5, 6, 12}
True False
hash(6) = 6 | hash('a') khác nhau giữa các lần chạy, nhưng hash số nguyên nhỏ = chính nó


### 2.4. Đo thời gian thô bằng `time.perf_counter`

Để *thấy* sự khác biệt list vs set (chỉ minh họa, con số phụ thuộc máy):

In [5]:
import time
big = list(range(100_000))
big_set = set(big)
t0 = time.perf_counter()
for _ in range(200):
    99_999 in big
t1 = time.perf_counter()
for _ in range(200):
    99_999 in big_set
t2 = time.perf_counter()
print(f"list: {t1 - t0:.4f}s | set: {t2 - t1:.6f}s -> set nhanh hơn rất nhiều")

list: 0.1182s | set: 0.000083s -> set nhanh hơn rất nhiều


### 2.5. Nhân đôi = dịch bit trái

`x * 2` bằng `x << 1` (dịch các bit sang trái 1 vị trí). `original · 2^k` bằng `original << k`. Ta sẽ dùng ở Cách 4.

In [6]:
x = 3
for k in range(4):
    print(k, x << k, x * 2 ** k, bin(x << k))

0 3 3 0b11
1 6 6 0b110
2 12 12 0b1100
3 24 24 0b11000


## 3. Cách 1 — Mô phỏng trực tiếp với `in list`

Đúng lời đề: `while original in nums: original *= 2`.

In [7]:
def find_final_list(nums: list[int], original: int) -> int:
    while original in nums:
        original *= 2
    return original

print(find_final_list([5, 3, 6, 1, 12], 3))   # 24
print(find_final_list([2, 7, 9], 4))          # 4

24
4


### Chi phí

- Số vòng `while`: giá trị đi `original, 2·original, 4·original, ...`; muốn còn "tìm thấy" thì phải ≤ 1000 → nhiều nhất khoảng `log2(1000 / original) + 1 ≤ 11` vòng. Gọi `M = 1000` là giá trị lớn nhất, số vòng là `O(log M)`.
- Mỗi vòng `in list` tốn `O(n)`.
- Tổng: **`O(n · log M)`** — với constraints thì rất nhỏ (≈ 11.000 phép so sánh).

Vậy Cách 1 **đã đủ nhanh**; nhưng thói quen tốt là nhận ra "kiểm tra tồn tại lặp lại" → dùng set.

In [8]:
import math
for orig in (1, 3, 100, 1000):
    print(f"original={orig:>4}: số vòng tối đa ≈ {math.floor(math.log2(1000 / orig)) + 1}")

original=   1: số vòng tối đa ≈ 10
original=   3: số vòng tối đa ≈ 9
original= 100: số vòng tối đa ≈ 4
original=1000: số vòng tối đa ≈ 1


## 4. Insight

**Nút thắt:** câu hỏi "giá trị `v` có trong `nums` không?" được hỏi lặp lại, mỗi lần dò lại cả list.

**Insight:** dữ liệu `nums` **không đổi** trong suốt quá trình, chỉ `original` thay đổi → **tiền xử lý một lần** thành `set`, sau đó mỗi câu hỏi `O(1)` trung bình.

Tư duy chung: *"Nếu cùng một tập dữ liệu bị hỏi 'có chứa x không?' nhiều lần, hãy đổ nó vào set."*

## 5. Cách 2 — Lời giải chính: set + while

Từng đoạn:

**Đoạn 1:** `seen = set(nums)` — `O(n)`.

**Đoạn 2:** `while original in seen:` — mỗi lần kiểm tra `O(1)` trung bình.

**Đoạn 3:** `original *= 2` (viết tắt của `original = original * 2`).

**Đoạn 4:** thoát vòng ⇒ `original` không còn trong `seen` ⇒ trả về.

In [9]:
def find_final_set(nums: list[int], original: int) -> int:
    seen = set(nums)              # Đoạn 1
    while original in seen:       # Đoạn 2
        original *= 2             # Đoạn 3
    return original               # Đoạn 4

print(find_final_set([5, 3, 6, 1, 12], 3))
print(find_final_set([2, 7, 9], 4))
print(find_final_set([1, 2, 4, 8, 16, 32, 64, 128, 256, 512], 1))

24
4
1024


## 6. TRACE từng bước

`nums = [5,3,6,1,12]` → `seen = {1,3,5,6,12}`, `original = 3`.

| Vòng | `original` trước | `original in seen`? | Hành động | `original` sau |
|:-:|:-:|:-:|---|:-:|
| 1 | 3 | ✓ | nhân đôi | 6 |
| 2 | 6 | ✓ | nhân đôi | 12 |
| 3 | 12 | ✓ | nhân đôi | 24 |
| 4 | 24 | ✗ | **dừng**, trả 24 | — |

In [10]:
def trace_find_final(nums, original):
    seen = set(nums)
    print("seen =", sorted(seen), "| original =", original)
    turn = 1
    while original in seen:
        print(f"Vòng {turn}: {original} có trong seen -> nhân đôi thành {original * 2}")
        original *= 2
        turn += 1
    print(f"Vòng {turn}: {original} KHÔNG có trong seen -> dừng, trả {original}")
    return original

trace_find_final([5, 3, 6, 1, 12], 3)

seen = [1, 3, 5, 6, 12] | original = 3
Vòng 1: 3 có trong seen -> nhân đôi thành 6
Vòng 2: 6 có trong seen -> nhân đôi thành 12
Vòng 3: 12 có trong seen -> nhân đôi thành 24
Vòng 4: 24 KHÔNG có trong seen -> dừng, trả 24


24

In [11]:
trace_find_final([2, 7, 9], 4)

seen = [2, 7, 9] | original = 4
Vòng 1: 4 KHÔNG có trong seen -> dừng, trả 4


4

## 7. Code nộp LeetCode — Cách 2

In [12]:
class Solution:
    def findFinalValue(self, nums: list[int], original: int) -> int:
        seen = set(nums)
        while original in seen:
            original *= 2
        return original

sol = Solution()
print(sol.findFinalValue([5, 3, 6, 1, 12], 3), sol.findFinalValue([2, 7, 9], 4))

24 4


## 8. Cách 3 — Sắp xếp rồi quét một lượt

Quan sát: chuỗi giá trị cần tìm `original < 2·original < 4·original < ...` là **tăng dần**. Nếu `nums` đã sắp tăng dần, ta quét từ trái sang phải; mỗi khi gặp đúng `x == original` thì nhân đôi. Vì chuỗi tăng, giá trị tiếp theo (nếu có) chắc chắn nằm **phía sau** → không cần quay lại.

In [13]:
def find_final_sort(nums: list[int], original: int) -> int:
    for x in sorted(nums):
        if x == original:
            original *= 2
    return original

print(find_final_sort([5, 3, 6, 1, 12], 3), find_final_sort([2, 7, 9], 4))

24 4


In [14]:
def trace_sort(nums, original):
    print("sorted:", sorted(nums))
    for x in sorted(nums):
        hit = x == original
        if hit:
            original *= 2
        print(f"x={x:>3}: {'khớp -> original=' + str(original) if hit else 'bỏ qua'}")
    return original

trace_sort([12, 6, 3, 3, 1, 5], 3)

sorted: [1, 3, 3, 5, 6, 12]
x=  1: bỏ qua
x=  3: khớp -> original=6
x=  3: bỏ qua
x=  5: bỏ qua
x=  6: khớp -> original=12
x= 12: khớp -> original=24


24

Trùng lặp như `3, 3` không gây lỗi: lần đầu gặp `3` ta đổi `original = 6`, lần sau `3 != 6` nên bỏ qua.

**Vì sao không bỏ sót?** Khi `original` được nhân đôi tại vị trí `x`, giá trị mới `2x > x` nên mọi phần tử bằng `2x` đều nằm **sau** vị trí hiện tại trong dãy tăng dần.

In [15]:
class Solution:
    def findFinalValue(self, nums: list[int], original: int) -> int:
        for x in sorted(nums):
            if x == original:
                original *= 2
        return original

print(Solution().findFinalValue([5, 3, 6, 1, 12], 3))

24


## 9. Cách 4 — Bitmask các lũy thừa của 2 (sáng tạo)

Mọi giá trị mà `original` có thể nhận đều có dạng `original · 2^k`. Ta duyệt `nums` **một lần**: nếu `x` có dạng đó thì bật bit `k` trong `mask`.

- `x % original == 0` và `q = x // original` là **lũy thừa của 2**.
- Kiểm tra lũy thừa của 2: `q & (q - 1) == 0` (lũy thừa của 2 chỉ có **một** bit 1).
- `k = q.bit_length() - 1`.

Đáp án là `original · 2^t` với `t` = **bit 0 thấp nhất** của `mask`. Mẹo tìm: `t = (~mask & (mask + 1)).bit_length() - 1`, hoặc đơn giản đếm bit 1 liên tiếp từ bit 0.

In [16]:
for q in (1, 2, 3, 4, 6, 8):
    print(q, bin(q), "| q & (q-1) =", q & (q - 1), "| lũy thừa 2?", q & (q - 1) == 0, "| bit_length-1 =", q.bit_length() - 1)

1 0b1 | q & (q-1) = 0 | lũy thừa 2? True | bit_length-1 = 0
2 0b10 | q & (q-1) = 0 | lũy thừa 2? True | bit_length-1 = 1
3 0b11 | q & (q-1) = 2 | lũy thừa 2? False | bit_length-1 = 1
4 0b100 | q & (q-1) = 0 | lũy thừa 2? True | bit_length-1 = 2
6 0b110 | q & (q-1) = 4 | lũy thừa 2? False | bit_length-1 = 2
8 0b1000 | q & (q-1) = 0 | lũy thừa 2? True | bit_length-1 = 3


In [17]:
def find_final_bitmask(nums: list[int], original: int) -> int:
    mask = 0
    for x in nums:
        if x % original == 0:
            q = x // original
            if q & (q - 1) == 0:              # q là lũy thừa của 2
                mask |= 1 << (q.bit_length() - 1)
    t = 0
    while mask >> t & 1:                       # đếm bit 1 liên tiếp từ bit 0
        t += 1
    return original << t

print(find_final_bitmask([5, 3, 6, 1, 12], 3), find_final_bitmask([2, 7, 9], 4))

24 4


In [18]:
def trace_bitmask(nums, original):
    mask = 0
    for x in nums:
        note = "bỏ"
        if x % original == 0:
            q = x // original
            if q & (q - 1) == 0:
                k = q.bit_length() - 1
                mask |= 1 << k
                note = f"= {original}·2^{k} -> bật bit {k}"
        print(f"x={x:>3}: {note:<24} mask={bin(mask)}")
    t = 0
    while mask >> t & 1:
        t += 1
    print("bit 0 thấp nhất ở vị trí", t, "-> đáp án", original << t)

trace_bitmask([5, 3, 6, 1, 12, 48], 3)

x=  5: bỏ                       mask=0b0
x=  3: = 3·2^0 -> bật bit 0     mask=0b1
x=  6: = 3·2^1 -> bật bit 1     mask=0b11
x=  1: bỏ                       mask=0b11
x= 12: = 3·2^2 -> bật bit 2     mask=0b111
x= 48: = 3·2^4 -> bật bit 4     mask=0b10111
bit 0 thấp nhất ở vị trí 3 -> đáp án 24


Trong ví dụ trên có `48 = 3·2^4` nhưng thiếu `24 = 3·2^3`, nên chuỗi dừng ở 24 — bit 4 bật cũng vô ích, đúng như mô phỏng.

In [19]:
class Solution:
    def findFinalValue(self, nums: list[int], original: int) -> int:
        mask = 0
        for x in nums:
            if x % original == 0:
                q = x // original
                if q & (q - 1) == 0:
                    mask |= 1 << (q.bit_length() - 1)
        t = (~mask & (mask + 1)).bit_length() - 1   # vị trí bit 0 thấp nhất
        return original << t

print(Solution().findFinalValue([5, 3, 6, 1, 12], 3), Solution().findFinalValue([2, 7, 9], 4))

24 4


Giải thích `~mask & (mask + 1)`: cộng 1 vào `mask` sẽ "lật" chuỗi bit 1 ở cuối thành 0 và bit 0 đầu tiên thành 1; `& ~mask` giữ lại đúng bit đó.

In [20]:
mask = 0b0111
print(bin(mask), bin(mask + 1), bin(~mask & (mask + 1)), "-> vị trí", (~mask & (mask + 1)).bit_length() - 1)

0b111 0b1000 0b1000 -> vị trí 3


## 10. Vì sao đúng? (invariant)

**Cách 2:** invariant trước mỗi lần kiểm tra `while`: `original` là giá trị hiện tại đúng như quá trình của đề (đã qua đúng các lần "tìm thấy" trước đó). Vì `seen` chứa đúng các giá trị của `nums`, `original in seen` ⟺ `original in nums`. Vòng dừng chính xác ở lần "không tìm thấy" đầu tiên. Vòng chắc chắn dừng vì `original` tăng gấp đôi và `seen` hữu hạn.

**Cách 3:** invariant khi quét dãy tăng dần tới phần tử `x`: mọi phần tử đứng trước `x` đều `< original` hoặc đã được "dùng". Phần tử bằng `original` (nếu có) chưa bị bỏ qua vì `original` chỉ tăng và luôn > các phần tử đã khớp.

**Cách 4:** bit `k` bật ⟺ `original·2^k` có trong `nums`. Quá trình đi qua `k = 0, 1, 2, ...` và dừng ở `k` đầu tiên mà bit tắt — chính là bit 0 thấp nhất.

## 11. Độ phức tạp

Gọi `n = len(nums)`, `M = 1000` (giá trị lớn nhất), số vòng nhân đôi `L = O(log M)`.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. `while` + `in list` | `O(n · log M)` | `O(1)` | đủ nhanh với constraints |
| 2. `set` + `while` | `O(n + log M)` trung bình | `O(n)` | lời giải chính |
| 3. Sắp xếp + quét | `O(n log n)` | `O(n)` (list sắp mới) | không cần hash |
| 4. Bitmask | `O(n)` | `O(1)` (mask ≤ ~10 bit) | sáng tạo, một lượt |

## 12. Bộ test: mẫu đề, biên, phản ví dụ, đối chiếu

1. Hai ví dụ của đề.
2. `original` không có trong `nums` (trả ngay).
3. Chuỗi rất dài: `1, 2, 4, ..., 512` → trả `1024`.
4. Có "lỗ hổng": chứa `3, 12` nhưng thiếu `6` → dừng ở `6`.
5. Phần tử trùng lặp.
6. Phần tử là bội của `original` nhưng không phải `original·2^k` (ví dụ `9` khi `original = 3`).

In [21]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<24} PASS ({actual})")

impls = [find_final_list, find_final_set, find_final_sort, find_final_bitmask, Solution().findFinalValue]
cases = [
    ("đề ví dụ 1", [5, 3, 6, 1, 12], 3, 24),
    ("đề ví dụ 2", [2, 7, 9], 4, 4),
    ("không có ngay", [1], 2, 2),
    ("chuỗi dài nhất", [1, 2, 4, 8, 16, 32, 64, 128, 256, 512], 1, 1024),
    ("có lỗ hổng", [3, 12, 24], 3, 6),
    ("trùng lặp", [3, 3, 6, 6], 3, 12),
    ("bội không phải 2^k", [3, 9, 27], 3, 6),
    ("original = 1000", [1000], 1000, 2000),
]
for name, nums, orig, exp in cases:
    for f in impls:
        assert f(list(nums), orig) == exp, (name, f)
    check(name, find_final_set(nums, orig), exp)

đề ví dụ 1               PASS (24)
đề ví dụ 2               PASS (4)
không có ngay            PASS (2)
chuỗi dài nhất           PASS (1024)
có lỗ hổng               PASS (6)
trùng lặp                PASS (12)
bội không phải 2^k       PASS (6)
original = 1000          PASS (2000)


In [22]:
import random
rng = random.Random(2154)
for _ in range(1000):
    orig = rng.choice([1, 2, 3, 5, 7, 10])
    pool = [orig << k for k in range(8)] + [rng.randint(1, 1000) for _ in range(5)]
    nums = [v for v in pool if v <= 1000 and rng.random() < 0.6] or [1]
    expected = find_final_list(nums, orig)
    for f in impls:
        assert f(list(nums), orig) == expected, (nums, orig, f)
print("1000 test ngẫu nhiên: mọi cách khớp nhau")

1000 test ngẫu nhiên: mọi cách khớp nhau


## 13. Lỗi người mới hay gặp

### Lỗi 1: Dùng `if` thay vì `while` (chỉ nhân đôi một lần)

In [23]:
def wrong_if(nums, original):
    if original in nums:
        original *= 2
    return original

print("sai :", wrong_if([5, 3, 6, 1, 12], 3), "| đúng:", find_final_set([5, 3, 6, 1, 12], 3))

sai : 6 | đúng: 24


**Vì sao sai:** đề yêu cầu **lặp lại** chừng nào còn tìm thấy. **Sửa:** `while`.

### Lỗi 2: Duyệt `nums` theo thứ tự gốc một lần (không sắp xếp)

In [24]:
def wrong_single_pass_unsorted(nums, original):
    for x in nums:
        if x == original:
            original *= 2
    return original

nums = [12, 6, 3]
print("sai :", wrong_single_pass_unsorted(nums, 3), "| đúng:", find_final_set(nums, 3))

sai : 6 | đúng: 24


**Vì sao sai:** `12` và `6` đứng **trước** `3`; khi đổi thành `6` ta đã đi qua `6` mất rồi. **Sửa:** sắp tăng dần trước (Cách 3) hoặc dùng set (Cách 2).

### Lỗi 3: Tạo set **bên trong** vòng lặp

In [25]:
def slow_but_correct(nums, original):
    while original in set(nums):      # mỗi vòng xây lại set: O(n) mỗi vòng
        original *= 2
    return original

print(slow_but_correct([5, 3, 6, 1, 12], 3), "<- đúng nhưng mất lợi ích của set")

24 <- đúng nhưng mất lợi ích của set


**Vì sao dở:** `set(nums)` tốn `O(n)` mỗi lần → còn chậm hơn `in list`. **Sửa:** tạo set **một lần** trước vòng lặp.

### Lỗi 4: Kiểm tra lũy thừa của 2 bằng `q % 2 == 0`

In [26]:
for q in (1, 6, 8):
    print(q, "q % 2 == 0:", q % 2 == 0, "| q & (q-1) == 0:", q & (q - 1) == 0)

1 q % 2 == 0: False | q & (q-1) == 0: True
6 q % 2 == 0: True | q & (q-1) == 0: False
8 q % 2 == 0: True | q & (q-1) == 0: True


**Vì sao sai:** `6` chẵn nhưng không phải lũy thừa của 2; còn `1 = 2^0` lẻ nhưng lại là lũy thừa của 2. **Sửa:** `q > 0 and q & (q - 1) == 0`.

## 14. Mẹo nhận dạng pattern

- "Lặp lại cho tới khi không còn tìm thấy" → `while` + kiểm tra tồn tại → **set**.
- Dữ liệu tĩnh, câu hỏi tồn tại lặp lại → tiền xử lý thành set/dict.
- Giá trị **tăng theo cấp số nhân** → số vòng chỉ `O(log M)`; đừng sợ `while`.
- Chuỗi giá trị đơn điệu + dữ liệu sắp xếp → có thể quét **một lượt**.
- Chỉ quan tâm các giá trị dạng `a·2^k` → bitmask theo `k`.

Bài liên quan: #128 Longest Consecutive Sequence (set + đi theo chuỗi), #217 Contains Duplicate, #1346 Check If N and Its Double Exist, #954 Array of Doubled Pairs.

## 15. Biến thể

### 15.1. Nhân với `m` bất kỳ (thay vì 2)

In [27]:
def find_final_multiplier(nums, original, m):
    seen = set(nums)
    while original in seen:
        original *= m
    return original

print(find_final_multiplier([2, 6, 18, 54], 2, 3))

162


Cẩn thận: nếu `m = 1` thì `original` không đổi → **lặp vô hạn** khi `original` có trong `nums`. Luôn xét điều kiện dừng khi đổi đề.

### 15.2. Trả về **độ dài chuỗi** (số lần nhân đôi) thay vì giá trị cuối

In [28]:
def count_doublings(nums, original):
    seen = set(nums)
    steps = 0
    while original in seen:
        original *= 2
        steps += 1
    return steps

print(count_doublings([5, 3, 6, 1, 12], 3))

3


### 15.3. LeetCode 1346 — Check If N and Its Double Exist
Có tồn tại `i != j` với `arr[i] == 2 * arr[j]`? Dùng set, chú ý số `0` (cần ít nhất hai số 0).

In [29]:
def check_if_exist(arr):
    seen = set()
    for x in arr:
        if 2 * x in seen or (x % 2 == 0 and x // 2 in seen):
            return True
        seen.add(x)
    return False

print(check_if_exist([10, 2, 5, 3]), check_if_exist([3, 1, 7, 11]), check_if_exist([0, 0]), check_if_exist([0]))

True False True False


## 16. Bài tập (TODO — tự làm, đáp án ở cuối)

### Bài 1 — Đoán output
1. `findFinalValue([8, 1, 2, 4], 1)` = ?
2. `findFinalValue([8, 1, 4], 1)` = ?
3. `find_final_bitmask` với `nums=[10, 20, 40, 30]`, `original=5`: `mask` cuối cùng ở dạng nhị phân là gì?

### Bài 2 — Đếm vòng
Với `original = 7` và `nums` chứa mọi số từ 1 đến 1000, vòng `while` chạy bao nhiêu lần và trả về gì?

### Bài 3 — Điền chỗ trống (Cách 2)

In [30]:
def exercise_find_final(nums: list[int], original: int) -> int:
    # TODO 1: tạo set từ nums
    # TODO 2: while original có trong set: nhân đôi
    # TODO 3: return original
    pass

# assert exercise_find_final([5, 3, 6, 1, 12], 3) == 24
# assert exercise_find_final([2, 7, 9], 4) == 4

### Bài 4 — Viết lại Cách 3 (sắp xếp) nhưng **dừng sớm** khi phần tử hiện tại đã lớn hơn `original`

In [31]:
def exercise_sort_early_stop(nums: list[int], original: int) -> int:
    # TODO
    pass

# assert exercise_sort_early_stop([5, 3, 6, 1, 12], 3) == 24
# assert exercise_sort_early_stop([3, 12, 24], 3) == 6

### Bài 5 — Biến thể: trả về **cả chuỗi** giá trị đã tìm thấy
Ví dụ `([5,3,6,1,12], 3)` → `[3, 6, 12]`.

In [32]:
def exercise_chain(nums: list[int], original: int) -> list[int]:
    # TODO
    pass

# assert exercise_chain([5, 3, 6, 1, 12], 3) == [3, 6, 12]
# assert exercise_chain([2, 7, 9], 4) == []

### Bài 6 — Mini challenge: chuỗi nhân đôi dài nhất
Với mỗi phần tử `x` của `nums` làm điểm xuất phát, tính số lần nhân đôi liên tiếp tìm thấy. Trả về độ dài lớn nhất. Yêu cầu: mỗi chuỗi chỉ đi từ **điểm đầu** (phần tử mà `x/2` không có trong `nums`) để tránh lặp lại công việc.

In [33]:
def exercise_longest_doubling_chain(nums: list[int]) -> int:
    # TODO: set; chỉ bắt đầu từ x mà (x % 2 == 1 hoặc x // 2 không có trong set)
    pass

# assert exercise_longest_doubling_chain([5, 3, 6, 1, 12]) == 3      # 3 -> 6 -> 12
# assert exercise_longest_doubling_chain([1, 2, 4, 8, 3]) == 4

### Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | Đi theo 1 → 2 → 4 → 8 | câu 2 thiếu số 2 | câu 3: 10=5·2, 20=5·4, 40=5·8 |
| 2 | 7·2^k ≤ 1000 | k = 0..7 (7·128 = 896) | 8 vòng, trả 7·256 |
| 3 | `seen = set(nums)` | `while original in seen` | `original *= 2` |
| 4 | Dãy tăng dần | `if x > original: break` | `elif x == original: original *= 2` |
| 5 | Ghi lại trước khi nhân | `chain.append(original)` | trong thân `while` |
| 6 | Giống #128 | điểm đầu: `x % 2 or x // 2 not in s` | đếm `while cur in s: cur *= 2; length += 1` |

## 17. Đáp án tham khảo (tự làm trước rồi hãy xem)

**Bài 1:** `16`; `2` (thiếu số 2); `mask = 0b1110` (bit 1, 2, 3 bật; `30` không phải `5·2^k`).

**Bài 2:** `7·2^k` với `k = 0..7` đều ≤ 1000 (`7·128 = 896`), `7·256 = 1792 > 1000` → chạy **8** vòng, trả **1792**.

In [34]:
print(Solution().findFinalValue([8, 1, 2, 4], 1), Solution().findFinalValue([8, 1, 4], 1))
trace_bitmask([10, 20, 40, 30], 5)
print(count_doublings(list(range(1, 1001)), 7), find_final_set(list(range(1, 1001)), 7))

16 2
x= 10: = 5·2^1 -> bật bit 1     mask=0b10
x= 20: = 5·2^2 -> bật bit 2     mask=0b110
x= 40: = 5·2^3 -> bật bit 3     mask=0b1110
x= 30: bỏ                       mask=0b1110
bit 0 thấp nhất ở vị trí 0 -> đáp án 5
8 1792


In [35]:
def answer_find_final(nums, original):
    seen = set(nums)
    while original in seen:
        original *= 2
    return original


def answer_sort_early_stop(nums, original):
    for x in sorted(nums):
        if x > original:
            break
        if x == original:
            original *= 2
    return original


def answer_chain(nums, original):
    seen = set(nums)
    chain = []
    while original in seen:
        chain.append(original)
        original *= 2
    return chain


def answer_longest_doubling_chain(nums):
    s = set(nums)
    best = 0
    for x in s:
        if x % 2 == 1 or x // 2 not in s:      # x là điểm đầu chuỗi
            cur, length = x, 0
            while cur in s:
                cur *= 2
                length += 1
            best = max(best, length)
    return best


assert answer_find_final([5, 3, 6, 1, 12], 3) == 24
assert answer_find_final([2, 7, 9], 4) == 4
assert answer_sort_early_stop([5, 3, 6, 1, 12], 3) == 24
assert answer_sort_early_stop([3, 12, 24], 3) == 6
assert answer_chain([5, 3, 6, 1, 12], 3) == [3, 6, 12]
assert answer_chain([2, 7, 9], 4) == []
assert answer_longest_doubling_chain([5, 3, 6, 1, 12]) == 3
assert answer_longest_doubling_chain([1, 2, 4, 8, 3]) == 4
for name, nums, orig, exp in cases:
    assert answer_sort_early_stop(nums, orig) == exp, name
print("Đáp án tham khảo: PASS toàn bộ test!")

Đáp án tham khảo: PASS toàn bộ test!


Lưu ý Bài 4: sau khi `original` nhân đôi tại `x`, phần tử kế tiếp có thể vẫn `< original` mới — chỉ khi `x > original` (đã vượt qua mà không khớp) mới chắc chắn không thể tìm thấy nữa, nên `break` đặt **trước** phép so sánh bằng.

## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def findFinalValue(self, nums: list[int], original: int) -> int:
        seen = set(nums)
        while original in seen:
            original *= 2
        return original
```

**Khung tư duy:** dữ liệu tĩnh + hỏi tồn tại nhiều lần → set một lần → `while` với điều kiện dừng chắc chắn.

**Checklist trước khi nộp:**
- [ ] Dùng `while` (lặp lại), không phải `if`?
- [ ] Tạo set **ngoài** vòng lặp?
- [ ] Vòng lặp chắc chắn dừng (giá trị tăng, tập hữu hạn)?
- [ ] Nếu quét một lượt, đã **sắp xếp** trước chưa?
- [ ] Test: không tìm thấy ngay, có lỗ hổng, trùng lặp.